In [2]:
import GEOparse

gse_tumor = GEOparse.get_GEO("GSE4271", destdir="../data/raw/geo",silent=True)
gse_normal = GEOparse.get_GEO("GSE6735", destdir="../data/raw/geo",silent=True)

In [18]:
import pandas as pd
import numpy as np

def build_expr_matrix(gse):
    exprs = []

    for gsm_name, gsm in gse.gsms.items():
        table = gsm.table

        expr = table[["ID_REF", "VALUE"]].copy()
        expr.columns = ["probe", gsm_name]

        exprs.append(expr)

    merged = exprs[0]

    for e in exprs[1:]:
        merged = merged.merge(e, on="probe")

    return merged.set_index("probe")

tumor_expr = build_expr_matrix(gse_tumor)
normal_expr = build_expr_matrix(gse_normal)

mmc2_df = pd.read_csv("../data/interim/mmc2_metadata.csv")
mmc2_df = mmc2_df.set_index("id")

In [19]:
tumor_expr = np.log2(tumor_expr + 1)
normal_expr = np.log2(normal_expr + 1)

if isinstance(normal_expr, pd.DataFrame):
    normal_expr["row_mean"] = normal_expr.mean(axis=1)
    normal_expr = normal_expr["row_mean"]

def choose_higher_ab(expression_df, metadata_df):
    selected = {}

    for patient_id, row in metadata_df[["A", "B"]].iterrows():
        sample_a = row["A"]
        sample_b = row["B"]

        if pd.isna(sample_a) and pd.isna(sample_b):
            continue

        if pd.notna(sample_a) and pd.notna(sample_b):
            if sample_a not in expression_df.columns or sample_b not in expression_df.columns:
                continue

            expr_a = expression_df[sample_a]
            expr_b = expression_df[sample_b]
            selected[patient_id] = pd.concat([expr_a, expr_b], axis=1).max(axis=1, skipna=True)
        elif pd.notna(sample_a) and sample_a in expression_df.columns:
            selected[patient_id] = expression_df[sample_a]
        elif pd.notna(sample_b) and sample_b in expression_df.columns:
            selected[patient_id] = expression_df[sample_b]

    return pd.DataFrame(selected)


chosen_expr = choose_higher_ab(tumor_expr, mmc2_df)

print("chosen_expr shape:", chosen_expr.shape)
print("chosen_expr nonzero count:", (chosen_expr != 0).sum().sum())
print("chosen_expr min/max:", chosen_expr.min().min(), chosen_expr.max().max())

chosen_expr.to_csv("../data/interim/tumor_expr_by_patient.csv")

chosen_expr shape: (168, 100)
chosen_expr nonzero count: 16800
chosen_expr min/max: 1.84799690655495 17.731069057150133


In [21]:
gpl1 = gse_tumor.gpls["GPL96"]
gpl2 = gse_tumor.gpls["GPL97"]

gene_id1 = gpl1.table[["ID", "Gene Symbol"]]
gene_id2 = gpl2.table[["ID", "Gene Symbol"]]

gene_id = pd.concat([gene_id1, gene_id2], axis=0)
gene_id = gene_id.rename(columns={"ID": "ID_REF"}).set_index("ID_REF")
gene_id = gene_id.copy()
gene_id["Gene Symbol"] = gene_id["Gene Symbol"].str.strip()
gene_id = gene_id[
    (gene_id["Gene Symbol"].notna()) &
    (gene_id["Gene Symbol"] != "") &
    (gene_id["Gene Symbol"] != "---")
]

gene_expr = chosen_expr.merge(
    gene_id,
    left_index=True,
    right_index=True,
    how="inner"
)

gene_expr = gene_expr.set_index("Gene Symbol")

def expand_multigene_probes(df):
    df = df.copy()
    df = df.reset_index().rename(columns={"index": "Gene Symbol"})
    df["Gene Symbol"] = df["Gene Symbol"].str.split(" /// ")
    df = df.explode("Gene Symbol")
    df["Gene Symbol"] = df["Gene Symbol"].str.strip()
    df = df[df["Gene Symbol"] != ""]
    df = df.set_index("Gene Symbol")
    return df


gene_expr = expand_multigene_probes(gene_expr)
gene_expr = gene_expr.groupby(level=0).max()

print("gene_expr shape:", gene_expr.shape)
print("gene_expr nonzero count:", (gene_expr != 0).sum().sum())
print("gene_expr min/max:", gene_expr.min().min(), gene_expr.max().max())

gene_expr.to_csv("../data/interim/tumor_expr_by_gene.csv")

gene_expr shape: (130, 100)
gene_expr nonzero count: 13000
gene_expr min/max: 8.11790278909403 17.232191365398826


In [22]:
gene_expr

,3744,3745,3746,3747,3748,3749,3751,3752,3753,3754,...,4785,4787,4788,4790,4791,4793,5063,5070,9907,9938
Gene Symbol,,,,,,,,,,,,,,,,,,,,,
ABCF1,11.704206,11.538528,12.417642,12.079818,12.577523,11.772232,11.969854,12.041625,12.326064,12.265175,...,12.445687,11.444653,11.832020,11.712656,12.435722,11.643180,11.494006,11.085273,11.409762,11.620266
ACTB,16.167165,16.221732,16.290595,15.400599,15.989847,15.335212,15.484046,15.833251,15.855504,15.846193,...,15.732080,15.575985,15.370320,15.581992,15.654921,15.131447,15.149783,15.154379,15.445112,15.447868
ANAPC5,11.907867,12.584869,11.777050,11.876786,12.171802,12.065685,12.350083,12.099512,12.377671,11.953251,...,11.902601,12.025347,12.497228,12.410239,12.589955,11.812578,11.928740,12.023269,11.709256,11.614296
ARF1,13.695718,13.819710,13.976788,13.493268,14.299101,13.268440,14.084102,14.013890,13.890464,13.890701,...,13.631268,13.864351,13.268513,12.710419,13.693138,13.409855,13.105990,13.666268,13.707823,13.500929
ARF3,12.781319,12.213712,12.770106,13.048095,12.773345,13.060189,13.207853,13.290004,12.916122,12.951157,...,12.770746,13.092229,12.833819,12.535713,12.179132,12.832297,13.279567,12.756556,13.588281,13.726740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TMED2,13.092988,13.999216,13.036483,12.791631,13.526291,11.931107,12.714052,13.296615,13.577712,13.314838,...,12.966253,13.143064,12.895253,12.322238,13.017713,12.797682,12.929055,13.041163,12.334553,13.348853
USP22,12.442347,11.834155,12.334162,12.592901,12.788555,12.604901,12.940185,12.593111,12.737902,12.558540,...,12.449742,12.629015,12.531820,12.389820,12.881573,12.248194,12.865675,12.574097,12.265117,12.011716
YY1,11.918453,12.251187,11.759306,10.909293,12.052568,11.248224,11.746430,11.965676,12.121372,12.090245,...,12.209453,11.987619,12.372130,11.883445,12.029908,12.292005,11.845725,11.559903,11.271930,11.922993
